# Job Listing Prioritization for Career Changers: EDA/SDA 2 (Non-Text Analysis)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

`EDA_SDA_1.ipynb` explored the data, defended the scope decisions, and built its final table (sections 10 and 11). This notebook does the **non-text** in-depth analysis: it checks the factors kept for now, separates ghost listings from desperate employers, analyzes each time scale, narrows the target, **computer and mathematical**, to the listings a career changer can get (our niche), pairs it with the team's backgrounds, and builds **its own final table**. The NLP work comes next, in `EDA_SDA_3.ipynb`; statistics in `EDA_SDA_4.ipynb`; then the `TimeSeries` and `ML` notebooks.

**Inputs from EDA_SDA_1** (decision tables in its section 10):
- English-language listings in the United States (listings with no language or country label dropped)
- Monthly horizon, April–September 2026
- Every listing's full follow-up: long-open listings kept, nothing capped
- All occupation groups and seniority levels, and all candidate factor columns, including reposts as a feature
- Career-change accessibility flags: `entry_seniority`, `years_required`, `low_experience_ask`, `entry_wording`, `changer_wording`, `degree_required`, `equivalent_experience`, and the first-pass `career_open` (EDA_SDA_1 Q1g)

**Conventions carried over:**
- Every decision is recorded with its reasoning.
- Data-quality fixes happen in cleaning; scope decisions are defended first, then applied at the end (section 4).
- Aggregates only (counts, shares, quantiles, model estimates); never `df.head()` or sample rows. Clear outputs before committing.


**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, and what's the scope? | Final table 1 |
| EDA_SDA_2 | Which listings do we study? Non-text checks, ghost listings, the career-change target | Niche table, non-text features |
| EDA_SDA_3 | What features does the text add? (NLP) | Niche table + text features |
| EDA_SDA_4 | Which features hold up? (statistics) | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing patterns and features |
| ML | How should listings be ranked for a career changer? | The prioritization model |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

### Clarifying questions
- Which long-open listings are ghost listings, and which are employers still trying to hire?
- Does source matter beyond company hiring volume, or is it company volume in disguise?
- How precise is freshness, given the crawler finds listings a median of 1.5 days late?
- Is pay disclosure driven by state pay-transparency laws or job type, rather than employer intent?
- Does high company volume mean more competition, or more openings to win?
- How are values missing: at random, or tied to groups (source, occupation group, state)?
- How much of computer and mathematical is left once listings a career changer can't get are hidden, and is it enough for depth?
- Do high-volume employers post more entry-level listings, so that volume also signals reachability?
- Which niche inside computer and mathematical (occupations × seniority × other dimensions) connects to the team's backgrounds and has enough data for depth?

### Roadmap

1. **Setup:** libraries and EDA_SDA_1's final table
2. **Analysis views:** which slice of the table each analysis uses, and why
3. **Checks and analyses** (non-text)
4. **EDA_SDA_2 final table:** decisions defended, then applied at the end, with a reason for each step
5. **Conclusion and next steps** (into EDA_SDA_3)

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, PART1_TABLE,
    as_list, bar_h, finish, km_curve, km_median, km_survival, pct, show, surv_at, use_chart_style,
)
use_chart_style()

In [ ]:
# 1b Load EDA_SDA_1's final table (built and saved in its section 11)
if not PART1_TABLE.exists():
    raise FileNotFoundError("Run EDA_SDA_1 section 11 first: it builds and saves the final table.")
part1 = pd.read_pickle(PART1_TABLE)
print("EDA_SDA_1 final table:", part1.shape)

## 2. Analysis views

Some analyses are skewed by part of the table. Each analysis states which **view** it uses:

| View | Rows | Use it for | Why |
|:--|:--|:--|:--|
| **All listings** | Everything in the final table, reposts included | Ranking listings as an applicant sees them; the repost features | Each listing is a row the app could show |
| **Originals only** | Reposts (`is_repost`) excluded | Counts, volumes, application windows, company concentration | A reposted job counted twice inflates volumes and splits one long-open job into two short ones |
| **With `posted`** | About 41% of listings | Day-of-week patterns; crawler lag | `first_seen` days follow the crawler's schedule |
| **With a salary** | About 43% of listings | Pay level and pay comparisons | Pay is missing for the rest |
| **Closed listings** | Not still open at the snapshot | Simple descriptive durations | Survival methods use every listing; this view is only for plain summaries |
| **Long-open listings** | Open for a long time (threshold set in 3.1) | Ghost listings vs. desperate employers | The question only applies to them |
| **Open to career changers** | `career_open`, or the stricter rule chosen in 3.11 | Accessibility comparisons, inside and outside the niche | Shows whether reachable listings behave differently |
| **The niche** | The career-change target, chosen in 3.6 | The segment analysis and the final table | Focus |

## 3. Checks and analyses

Record each result and the decision it leads to here, then apply it when building the final table (section 4).

### 3.1 Ghost listings vs. desperate employers

A long-open listing may be a **ghost listing** (forgotten or no longer monitored: rank lower or flag it) or a **desperate employer** (still actively hiring: rank higher). Age alone can't tell them apart.

| Signal | Points to "still hiring" | Points to "ghost listing" |
|:--|:--|:--|
| Relisted (`times_listed`, `days_since_repost_last_listed`) | Relisted recently | Never relisted in months |
| Company still posting new listings | The company is actively recruiting | The company stopped posting long ago |
| Source | Hiring platforms often auto-expire listings, so a long-open one was likely kept on purpose | Company career pages can host forgotten listings indefinitely |
| `posted` vs. `first_seen` | Recently refreshed | Very old posting date |
| `last_processed_at` (meaning to confirm with PredictLeads) | May record content changes | |

Steps: describe the long-open listings (how many, which occupation groups and sources); measure each signal; combine them into a **"likely ghost listing" flag**; decide whether to filter, rank lower, or show the flag in the app. Evergreen wording ("always accepting applications") is added in EDA_SDA_3.

### 3.2 Source / platform

| Check | If the result is… | Then… |
|:--|:--|:--|
| Median days open by source, *within* each company-volume band | Source gaps hold within every band | Keep source as its own factor |
| | Gaps mostly disappear | Drop it, or keep it as a minor factor |
| Share of job-board listings whose company + title also appears on a company site or platform | High (copies) | Drop job-board rows to avoid double-counting |
| | Low | Keep them |
| Platform sizes in the chosen groups | Only a few platforms have 1,000+ listings | Name those platforms; group the rest as "other platform" |

### 3.3 Freshness and crawler lag

| Check | If the result is… | Then… |
|:--|:--|:--|
| Share of listings found more than 7, 14 and 30 days after `posted` | Tiny | Use `first_seen` only |
| | Meaningful | Add a "found late" flag |
| Survival curves: found late vs. found promptly | Found-late listings close much sooner | The flag matters for apply-by estimates |
| Survival curves: listings with vs. without a `posted` date | Similar | `posted` can be used where present |
| | Different | Stick with `first_seen` only |

### 3.4 Pay disclosed

| Check | If the result is… | Then… |
|:--|:--|:--|
| Disclosure rate by US state (needs the state from `location_data`) | Much higher in pay-transparency states (CA, CO, NY, WA) | Add state, or a "pay-law state" flag |
| Disclosure by hourly vs. salaried, and by seniority | Strongly tied to hourly | Its effect may really be job type |
| Days open by disclosure, within company-volume bands | The gap holds | Keep as a factor |
| | The gap disappears | Keep only as the companion flag for pay level |

### 3.5 Reposts and company hiring volume

- **Reposts** (a decided feature: how strongly should it count?): days open by `times_listed`; repost rate in the chosen groups; repost rate within company-volume bands, to see whether it adds anything beyond company volume.
- **Company hiring volume:** does high volume mean **more competition, or more openings to win?** Its direction in the ranking depends on the answer. Count each company's listings from before each listing only, so the future isn't used. The companies table has no company size or employee count, so listing volume is our only measure of how much a company hires.
- **For career changers:** do high-volume employers post a larger share of entry-level listings or training wording? If so, volume also signals that a newcomer can get in.

### 3.6 Choosing the career-change target (our niche)

**Decided (2026-10-09):** the target is **computer and mathematical (US)**, the source backgrounds are the team's own, and listings a career changer can't get are hidden (3.11). This section picks the niche *inside* computer and mathematical: a combination of occupations, seniority and other dimensions that a real career changer would search as one set, and could realistically get. *Examples only:* data analyst, IT support or QA roles for someone coming from customer service or operations. The goal is the best grouping **that also makes sense for someone switching roles or fields**.

**How narrow?** A niche can combine anything from one dimension (broad, barely a niche: "all computer and mathematical roles") to every dimension (extreme: occupation group + seniority + category + work arrangement). The project guidelines encourage a narrow scope, so **we look for balance, leaning a bit toward narrow: as narrow as still makes sense and still has enough data (below).** *Why:* a narrow niche gives specific, credible findings and a clear app audience, but too narrow leaves too little data for in-depth analysis.

**Dimensions a niche can combine:**

| Dimension | Field | Notes |
|:--|:--|:--|
| Occupation group(s) | `occupation_group` | Computer and mathematical (decided); the team's source-background groups kept for comparison and role discovery |
| Finer occupation | `occupation` | The main way to narrow inside computer and mathematical (for example data analysts, support specialists) |
| Seniority band | `seniority` | For example junior; mid/senior; manager and above |
| Category | `categories` | PredictLeads' 27 labels, such as "data_analysis" |
| Work arrangement | `contract_types`, `mentions_remote` | Full time, part time, remote, hybrid |

**Test 1: does it make sense?** A niche passes only if:
- **One applicant would search it.** The roles share skills and a career level, so the same résumé could fit most of them.
- **A career changer could get it.** A real share of listings is entry level or asks for 2 years or fewer (EDA_SDA_1 Q1g), and most don't require a license or a field-specific degree.
- **It connects to the team's backgrounds.** It shares skills with at least one teammate's background, judged with O*NET's related occupations and skills.
- **It's coherent.** Similar seniority, overlapping `categories`, and a pay range that isn't wildly spread.
- **It has a clear user story.** For example, "a customer service worker moving into IT support."
- **It's useful to the team and the app.** It has an audience, and team members' own backgrounds can serve as source backgrounds, which helps the later transferable-fit work.

**Test 2: is it analytically strong?** Measured on originals only:

| Criterion | Why |
|:--|:--|
| Listings, and listings still open at the snapshot | Enough to model, and enough to rank on any given day |
| Companies, and the top company's share | No single employer dominates |
| Variation in the factors (window, source, company volume, reposts, pay) **within** the niche | Prioritization needs differences to rank on |
| Coverage of key fields (description, salary, `posted`) | Factors and text features work for most listings |
| Words per description (EDA_SDA_1 Q3e) | Enough text for the NLP in Part 3 |
| Openness to career changers (`entry_share`, `career_open_share`, years asked, degree required) and pay | The niche must be reachable and worth the move |

**Is there enough data?** Not fixed thresholds for now, but questions to weigh against how narrow the niche is. The narrower the niche, the more these matter:
- **Enough listings overall** that patterns aren't flukes from a handful of companies?
- **Enough listings still open** to rank on any given day, as the app would?
- **Enough listings in each month** to compare months?
- **Enough companies** that no single employer's habits drive the results?
- **Enough listings with a description** for the NLP in Part 3?
- **Enough listings open to career changers** to rank within the niche, not just around it?

If a candidate niche is so narrow that any answer is "no", broaden it by one dimension.

**Measured depth check.** For each candidate niche:
1. List the splits the in-depth analysis will make (for example, 6 months × 3 source types = 18 sub-groups).
2. Measure each sub-group's size, on originals only.
3. Measure the 95% confidence-interval width of the key estimates in each sub-group: the share still open after 7 days, and the median days open.

A niche has **enough depth** if those intervals are about **±5 percentage points or narrower**: tight enough to show the 5–16 point differences the EDA found. Sub-groups wider than that are combined or dropped from the split.

**A rough guide to narrowness:** aim for two or three dimensions combined (about 50–75% of the way from no niche to the most extreme), as long as the depth check passes. In EDA_SDA_1's data, two dimensions keep roughly 1–11% of listings (thousands of listings each); three dimensions are workable only for common combinations.

**Procedure:**
1. **List 5–10 candidate niches** that pass Test 1, each with a one-line user story, *before* scoring them.
2. **Fix the Test 2 criteria** (what we'll compare) **before** looking at any model results. Choosing the niche where results look best would make the findings partly luck.
3. **Score every candidate,** in a table with one row per niche.
4. **Choose one niche,** balancing narrowness against the data questions above, leaning a bit toward narrow, plus a **second as the robustness check:** findings should hold in both.
5. **Record the reasoning:** why this niche, why not the others, and why not narrower or broader.
6. **Save the niche as its own filtered table,** the base for this notebook's final table (section 4) and for EDA_SDA_3.

**How related groups can be defined** (for niches spanning several occupation groups):
1. Shared PredictLeads `categories`.
2. O*NET's related-occupations data (public, outside our dataset).
3. Judgment, backed by Test 1.
4. Text similarity (TF-IDF), once EDA_SDA_3 exists; revisit then.
5. Skill overlap with the chosen source background (O*NET skills), so the target is reachable from where the career changer starts.

### 3.7 Blind spots from Part 1

| Blind spot | Check or fix |
|:--|:--|
| The "closed" outcome is wrong for reposted jobs: the earlier listing disappears without being filled | In the analysis (not as a model input), treat a listing that was later reposted as "closed without filling" or censored |
| The 2-day "still open" tolerance is a guess | Distribution of days between `last_seen` and the snapshot; a bump near 7 days means it's too tight |
| Annualizing hourly pay assumes full time | Adjust with `contract_types` (part time vs. full time), or flag part-time annualized pay |
| Company volume counted within the sample | The companies table has no size or employee count. It does have a description (industry, via NLP in EDA_SDA_3), a ticker (publicly traded), parent and subsidiary companies, and similar companies: check whether these add anything beyond volume |
| State isn't in the table | Add it from `location_data`, if PredictLeads includes it |
| Missing country vs. remote, missing language vs. description | Confirm the EDA_SDA_1 section 10a results |
| `entry_seniority` counts only the `junior` label | Check every seniority label (EDA_SDA_1 Q3b) and add any that are entry level, such as intern, to `ENTRY_LEVELS` |

### 3.8 Joining the companies table

Merge the companies table into the listings (a **left join** on `company_id` = company `id`, so every listing stays). Course standard: one master table from all sources, with keys checked first. Join checks: company ids are unique in the companies table (no row multiplication), and match rate (100% in EDA_SDA_1). Columns to bring in: `ticker` (publicly traded), parent company, subsidiary and similar companies, location, and description (its text goes to EDA_SDA_3). The table has **no company size or employee count**.

### 3.9 Missing-data patterns

Course standard: percent missing, missingness **by group** (source, occupation group, month, state), and `df.isnull().corr()` to see which fields go missing together. Advanced: decide each field's mechanism (missing completely at random, at random given other fields, or not at random) and handle it accordingly. Salary is likely *not* at random, because of pay-transparency laws. Every imputation or "Unknown" category is justified with plots split by the outcome (closed vs. still open).

### 3.10 Outliers

Boxplots and IQR rules for days open, words per description, annual salary, times listed. Course standard: inspect, explain, and keep or remove with a reason. Advanced: robust (median / IQR) summaries, and log scales for the right-skewed measures.

### 3.11 Career-change accessibility, and hiding what a career changer can't get

**Decided (2026-10-09): listings a career changer can't get are hidden.** Three kinds of blocker, each hidden where it's reliable:

| Blocker | Example | Hidden in |
|:--|:--|:--|
| Seniority | Manager level and above | This notebook's final table (the label is reliable). Check which labels count, from EDA_SDA_1 Q3b |
| Text, for everyone | Too many years of experience asked; a required license | EDA_SDA_3's final table, after the rules are validated |
| Text, per applicant | A degree or certification this applicant doesn't have | The app, per profile |

Report how many listings each rule hides. Then check:

| Check | If the result is… | Then… |
|:--|:--|:--|
| Accessibility flags by company-volume band and by source | Entry-level listings cluster at high-volume employers or on certain platforms | Account for it, so accessibility isn't company size or source in disguise |
| Pay of listings open to career changers vs. the rest, within the niche | A large pay gap | Show it in the app, and let the pay weight trade off against accessibility |
| Days open by accessibility, within the niche | Reachable listings close faster | Raise their urgency in the ranking |
| Degree required by occupation group | Near-universal in a group, with little "or equivalent experience" | That group is a poor target |
| Each part of `career_open` separately | One part (for example, entry-level wording) adds little | Use a stricter rule for the "open to career changers" view |

## 4. EDA_SDA_2 final table

_To do after section 3:_ defend each decision with its result from section 3, then apply them all at once, at the end: the niche inside computer and mathematical, hiding manager-level listings, the additional cleaning, each factor kept or dropped, and the ghost-listing decision. Record every step with its reason, in the same format as EDA_SDA_1 section 10 and 11. Save it as a separate file, so EDA_SDA_1's table stays unchanged.

## 5. Conclusion and next steps

_To do._ Next: `EDA_SDA_3.ipynb`, the NLP features, built on this notebook's niche table.